# Profiling 分析：Varlen+FSDP 4096 vs Varlen+CP 8192

本节读取 08.03 的等 token trace，检查 FSDP AllGather/ReduceScatter、CP AllToAll、TND shape、关键路径和峰值显存。分析前先重放 DataLoader：只有 trace 里的隔离区间数（真实样本与末尾 padding）与当前重放结果一致，才允许比较性能。最终吞吐需按 raw samples、input tokens 和 supervised tokens 归一化。


## 运行环境

先切到 TorchTitan-NPU 根目录；后续所有 trace 读取和结果导出都由 notebook cell 完成。


本 notebook 可以读取旧的单 rank trace 来检查执行路径，但只有同时找到 rank 0 和 rank 1 时才允许形成性能结论。所有分析和 CSV 导出均可在 cell 内重跑。

## 关闭 profiler 的稳态吞吐

先读取本轮 steps 2–12 的日志与逐批 DataLoader 重放。Step 1 含编译，不进入统计；四种吞吐都用 11 步累计分子除以累计 wall time，不用单个 batch 外推。


In [ ]:
import os
from pathlib import Path

original_dir = Path.cwd()
configured_root = os.environ.get('TORCHTITAN_ROOT')
candidates = [Path(configured_root)] if configured_root else []
for parent in (original_dir, *original_dir.parents):
    candidates.extend((parent / 'torchtitan-npu', parent.parent / 'torchtitan-npu'))
torchtitan_root = next((path.resolve() for path in candidates if (path / 'scripts/run_train.sh').is_file()), None)
if torchtitan_root is None:
    raise RuntimeError('未找到 torchtitan-npu；请设置 TORCHTITAN_ROOT。')
os.chdir(torchtitan_root)
if cann_env := os.environ.get('CANN_ENV_SCRIPT'):
    os.environ['BASH_ENV'] = cann_env
print('torchtitan root:', torchtitan_root)


In [ ]:
import json
import math
import os
import re
import statistics
from pathlib import Path

TORCHTITAN_ROOT = torchtitan_root
OUTPUT_ROOT = TORCHTITAN_ROOT / 'outputs'
EVIDENCE_ROOT = OUTPUT_ROOT / 'profile_traces/08_positions_v2_evidence'
STEADY_WORKLOAD = json.loads((EVIDENCE_ROOT / 'ch8_steady_workload.json').read_text())
STEADY_LOGS = {
    'varlen_fsdp': EVIDENCE_ROOT / '08_varlen_fsdp_s4096_positions_v2_no_prof.log',
    'varlen_cp': EVIDENCE_ROOT / '08_varlen_cp_s8192_positions_v2_no_prof.log',
}

def steady_log_metrics(path: Path, workload_rows: list[dict[str, int]]) -> dict[str, object]:
    text = path.read_text(errors='replace')
    step_times = [
        (int(step), float(seconds))
        for step, seconds in re.findall(
            r'step:\s*(\d+).*?elapsed_time_per_step:\s*([0-9.]+)s', text
        )
    ]
    # Step 1 includes compilation.  Steps 2--12 are the fixed steady window.
    times = [seconds for step, seconds in step_times if 2 <= step <= 12]
    rows_2_12 = workload_rows[1:12]
    if len(times) != len(rows_2_12) or len(times) != 11:
        raise ValueError(f'expected 11 steady steps, got times={len(times)}, rows={len(rows_2_12)}')
    fields = ('container_tokens', 'nonpadding_tokens', 'supervised_tokens', 'raw_samples')
    totals = {field: sum(row[field] for row in rows_2_12) for field in fields}
    wall_seconds = sum(times)
    memory_values = [float(value) for value in re.findall(r'memory:\s*([0-9.]+)GiB', text)]
    return {
        'steps': '2-12',
        'count': len(times),
        'mean_step_s': statistics.mean(times),
        'median_step_s': statistics.median(times),
        'p95_step_s': sorted(times)[math.ceil(0.95 * len(times)) - 1],
        'min_step_s': min(times),
        'max_step_s': max(times),
        'wall_seconds': wall_seconds,
        'max_reserved_gib': max(memory_values),
        'totals': totals,
        'rates_per_s': {field: value / wall_seconds for field, value in totals.items()},
    }

STEADY_RESULTS = {
    route: steady_log_metrics(STEADY_LOGS[route], STEADY_WORKLOAD[key])
    for route, key in (('varlen_fsdp', 'fsdp'), ('varlen_cp', 'cp'))
}

print('route | median/P95 step s | container tok/s | non-padding tok/s | supervised tok/s | raw sample/s | max reserved GiB')
for route, result in STEADY_RESULTS.items():
    rate = result['rates_per_s']
    print(
        f"{route} | {result['median_step_s']:.3f}/{result['p95_step_s']:.3f} | "
        f"{rate['container_tokens']:.1f} | {rate['nonpadding_tokens']:.1f} | "
        f"{rate['supervised_tokens']:.1f} | {rate['raw_samples']:.3f} | "
        f"{result['max_reserved_gib']:.2f}"
    )

print('CP relative to FSDP')
for metric in ('container_tokens', 'nonpadding_tokens', 'supervised_tokens', 'raw_samples'):
    fsdp_rate = STEADY_RESULTS['varlen_fsdp']['rates_per_s'][metric]
    cp_rate = STEADY_RESULTS['varlen_cp']['rates_per_s'][metric]
    print(f'{metric}: {cp_rate / fsdp_rate - 1:+.1%}')
print(
    'median step time:',
    f"{STEADY_RESULTS['varlen_cp']['median_step_s'] / STEADY_RESULTS['varlen_fsdp']['median_step_s'] - 1:+.1%}",
)

steady_report_path = OUTPUT_ROOT / 'profile_traces/08_positions_v2_steady_summary.json'
steady_report_path.write_text(json.dumps(STEADY_RESULTS, indent=2), encoding='utf-8')
print('saved:', steady_report_path)


## 1. 算子执行时间与收益

以 `Device Self Duration` 表示叶子算子自身占用 device 的时间；`Device Total Duration` 可包含子算子，不跨层级相加。以下单元汇总全部算子的 calls、total、avg、P50 和 P95，并导出完整 CSV。

In [ ]:
from __future__ import annotations

import csv
import math
import os
from collections import defaultdict
from pathlib import Path

TORCHTITAN_ROOT = torchtitan_root
SUFFIX = os.environ.get('TORCHTITAN_RUN_SUFFIX', '_positions_v2')
OUTPUT_ROOT = TORCHTITAN_ROOT / 'outputs'
TRACE_BASES = {
    'varlen_fsdp': '08_varlen_fsdp_s4096',
    'varlen_cp': '08_varlen_cp_s8192',
}

def select_trace_root(base: str) -> Path:
    names = [f'{base}{SUFFIX}'] if SUFFIX else [f'{base}_allranks', base]
    candidates = [OUTPUT_ROOT / 'profile_traces' / name for name in names]
    return next((path for path in candidates if path.exists()), candidates[0])

def discover_rank_outputs(root: Path) -> dict[int, Path]:
    result = {}
    for output in sorted(root.glob('*_ascend_pt/ASCEND_PROFILER_OUTPUT')):
        rank_dbs = sorted(output.glob('ascend_pytorch_profiler_*.db'))
        if rank_dbs:
            rank = int(rank_dbs[0].stem.rsplit('_', 1)[1])
            result[rank] = output
    return result

TRACE_ROOTS = {route: select_trace_root(base) for route, base in TRACE_BASES.items()}
TRACE_OUTPUTS_BY_RANK = {
    route: discover_rank_outputs(root) for route, root in TRACE_ROOTS.items()
}
REQUIRED_RANKS = {0, 1}
RANK_COVERAGE_CURRENT = all(
    set(outputs) == REQUIRED_RANKS for outputs in TRACE_OUTPUTS_BY_RANK.values()
)
for route, outputs in TRACE_OUTPUTS_BY_RANK.items():
    missing_ranks = REQUIRED_RANKS - set(outputs)
    if missing_ranks:
        print(f'{route}: missing ranks {sorted(missing_ranks)}; performance comparison is disabled')

def rows(path: Path) -> list[dict[str, str]]:
    with path.open(encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))

def field(row: dict[str, str], *names: str) -> str:
    for name in names:
        if row.get(name) not in (None, ''):
            return str(row[name])
    return ''

def number(row: dict[str, str], *names: str) -> float:
    value = field(row, *names).replace(',', '')
    try:
        return float(value)
    except ValueError:
        return 0.0

DURATION_COLUMNS = {
    'host_self': 'Host Self Duration(us)',
    'host_total': 'Host Total Duration(us)',
    'device_self': 'Device Self Duration(us)',
    'device_total': 'Device Total Duration(us)',
}

def percentile(values: list[float], q: float) -> float:
    if not values:
        return 0.0
    ordered = sorted(values)
    return ordered[max(0, math.ceil(q * len(ordered)) - 1)]

def aggregate_operators(output: Path) -> dict[str, dict[str, object]]:
    path = output / 'operator_details.csv'
    if not path.exists():
        return {}
    grouped = defaultdict(lambda: {
        'calls': 0, 'shapes': set(),
        **{metric: [] for metric in DURATION_COLUMNS},
    })
    for row in rows(path):
        item = grouped[row['Name']]
        item['calls'] += 1
        if field(row, 'Input Shapes'):
            item['shapes'].add(field(row, 'Input Shapes'))
        for metric, column in DURATION_COLUMNS.items():
            item[metric].append(number(row, column))
    result = {}
    for name, item in grouped.items():
        record = {'calls': item['calls'], 'shape_variants': len(item['shapes'])}
        for metric in DURATION_COLUMNS:
            values = [value for value in item[metric] if value > 0]
            record[f'{metric}_calls'] = len(values)
            record[f'{metric}_total_ms'] = sum(values) / 1000
            record[f'{metric}_avg_us'] = sum(values) / len(values) if values else 0.0
            record[f'{metric}_p50_us'] = percentile(values, 0.50)
            record[f'{metric}_p95_us'] = percentile(values, 0.95)
        result[name] = record
    return result

OPERATOR_STATS_BY_RANK = {
    route: {rank: aggregate_operators(output) for rank, output in outputs.items()}
    for route, outputs in TRACE_OUTPUTS_BY_RANK.items()
}
OPERATOR_ROWS = []
for route, rank_stats in OPERATOR_STATS_BY_RANK.items():
    for rank, operators in rank_stats.items():
        for operator, stats in operators.items():
            OPERATOR_ROWS.append({'route': route, 'rank': rank, 'operator': operator, **stats})
OPERATOR_ROWS.sort(key=lambda row: (row['route'], row['rank'], -row['device_self_total_ms']))
OPERATOR_REPORT = OUTPUT_ROOT / 'profile_traces/08_operator_profile_by_rank.csv'
if OPERATOR_ROWS:
    with OPERATOR_REPORT.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(OPERATOR_ROWS[0]))
        writer.writeheader()
        writer.writerows(OPERATOR_ROWS)

KEY_OPERATORS = (
    'aclnnMatmul',
    'aclnnFlashAttentionVarLenScore',
    'aclnnFlashAttentionUnpaddingScoreGrad',
    'HcclAllGather',
    'HcclReduceScatter',
    'HcclAlltoAllV',
)
print('route/rank | operator | calls | device self total ms | avg us | P95 us')
for route, rank_stats in OPERATOR_STATS_BY_RANK.items():
    for rank, operators in rank_stats.items():
        for operator in KEY_OPERATORS:
            stats = operators.get(operator)
            if stats:
                print(
                    f"{route}/{rank} | {operator} | {stats['device_self_calls']} | "
                    f"{stats['device_self_total_ms']:.3f} | {stats['device_self_avg_us']:.1f} | "
                    f"{stats['device_self_p95_us']:.1f}"
                )
print(f'\n全部 {len(OPERATOR_ROWS)} 条逐 rank 算子统计：{OPERATOR_REPORT}')


## 2. 预期路径

| route | trace 中至少应寻找 | 额外 Varlen+CP 证据 |
|---|---|---|
| Varlen+FSDP 4096 | all-gather + reduce-scatter | TND Varlen kernel |
| Varlen+CP 8192 | all-to-all pre/post | TND `[16384, 8, 128]`、`actual_seq_*`、`sparse_mode=7` |

如果只看到 Python 名称而没有 device event，不能确认 kernel；如果只看到 all-to-all 而没有 Varlen kernel，不能确认 Varlen+CP backend。

In [ ]:
import json
EXPECTED = {
    'varlen_fsdp': {'all_gather', 'reduce_scatter'},
    'varlen_cp': {'all_to_all'},
}

def kind(name: str) -> str | None:
    text = name.lower().replace(' ', '')
    if 'allgather' in text or 'all-gather' in text:
        return 'all_gather'
    if 'reducescatter' in text or 'reduce-scatter' in text:
        return 'reduce_scatter'
    if 'allreduce' in text or 'all-reduce' in text:
        return 'all_reduce'
    if 'alltoall' in text or 'all-to-all' in text:
        return 'all_to_all'
    if 'hcom' in text or 'hccl' in text or 'collective' in text:
        return 'other_collective'
    return None


In [ ]:
def parse_trace(root: Path) -> dict[str, object]:
    report: dict[str, object] = {
        'status': 'missing',
        'root': str(root),
        'operator_files': [],
        'trace_files': [],
        'memory_files': [],
        'communications': {},
        'communication_engine': {},
        'kernel_hits': {},
        'memory_peak_raw': None,
    }
    if not root.exists():
        return report
    report['status'] = 'present'
    detail_files = sorted(root.rglob('operator_details.csv'))
    trace_files = sorted(root.rglob('trace_view.json'))
    memory_files = sorted(root.rglob('memory_record.csv'))
    report['operator_files'] = [str(p) for p in detail_files]
    report['trace_files'] = [str(p) for p in trace_files]
    report['memory_files'] = [str(p) for p in memory_files]
    communication = defaultdict(lambda: {'calls': 0, 'device_self_us': 0.0})
    for path in detail_files:
        for row in rows(path):
            op_name = field(row, 'Name', 'Operator Name', 'Kernel Name')
            op_kind = kind(op_name)
            device_self_us = number(row, 'Device Self Duration(us)')
            if op_kind is None or device_self_us <= 0:
                continue
            communication[op_kind]['calls'] += 1
            communication[op_kind]['device_self_us'] += device_self_us
    report['communications'] = dict(communication)
    engine = defaultdict(lambda: {
        'calls': 0, 'zero_payload_calls': 0, 'elapsed_ms': 0.0,
        'wait_ms': 0.0, 'sync_ms': 0.0, 'link_transit_ms': 0.0, 'transit_mb': 0.0,
    })
    for path in sorted(root.rglob('communication.json')):
        try:
            steps = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        for step in steps.values():
            for op_name, event in step.get('collective', {}).items():
                op_kind = kind(op_name)
                if op_kind is None:
                    continue
                timing = event.get('Communication Time Info', {})
                bandwidth = event.get('Communication Bandwidth Info', {})
                item = engine[op_kind]
                link_sizes = [float(channel.get('Transit Size(MB)', 0)) for channel in bandwidth.values()]
                link_times = [float(channel.get('Transit Time(ms)', 0)) for channel in bandwidth.values()]
                item['calls'] += 1
                item['elapsed_ms'] += float(timing.get('Elapse Time(ms)', 0))
                item['wait_ms'] += float(timing.get('Wait Time(ms)', 0))
                item['sync_ms'] += float(timing.get('Synchronization Time(ms)', 0))
                item['link_transit_ms'] += max(link_times, default=0.0)
                payload = max(link_sizes, default=0.0)
                item['transit_mb'] += payload
                item['zero_payload_calls'] += int(payload == 0)
    report['communication_engine'] = dict(engine)
    hits = defaultdict(int)
    for path in trace_files:
        try:
            raw = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        events = raw if isinstance(raw, list) else raw.get('traceEvents', [])
        for event in events:
            text = json.dumps(event, ensure_ascii=False).lower()
            name = str(event.get('name', '')).lower()
            if 'npu_fusion_attention_v3' in text:
                hits['npu_fusion_attention_v3'] += 1
            if 'input_layout' in text and 'tnd' in text:
                hits['TND'] += 1
            if 'actual_seq_qlen' in text or 'actual_seq_kvlen' in text:
                hits['actual_seq_lengths'] += 1
            if 'sparse_mode' in text and ('7' in text or '7.0' in text):
                hits['sparse_mode_7'] += 1
            if 'alltoall' in name or 'all-to-all' in name:
                hits['trace_all_to_all'] += 1
    report['kernel_hits'] = dict(hits)
    memory_values = []
    for path in memory_files:
        for row in rows(path):
            for key, value in row.items():
                if key and any(word in key.lower() for word in ('peak', 'allocated', 'reserved')):
                    try:
                        memory_values.append(float(str(value).replace(',', '')))
                    except (TypeError, ValueError):
                        pass
    if memory_values:
        report['memory_peak_raw'] = max(memory_values)
    return report

REPORTS_BY_RANK = {
    route: {rank: parse_trace(output) for rank, output in outputs.items()}
    for route, outputs in TRACE_OUTPUTS_BY_RANK.items()
}
for route, rank_reports in REPORTS_BY_RANK.items():
    for rank, report in rank_reports.items():
        print(f'\n[{route}/rank{rank}] {report["status"]}: {report["root"]}')
        print('  communication engine:', report['communication_engine'] or '<none>')
        print('  Varlen/CP hits:', report['kernel_hits'] or '<none>')


In [ ]:
for route, rank_reports in REPORTS_BY_RANK.items():
    if not rank_reports:
        raise FileNotFoundError(f'{route}: no profiler output under {TRACE_ROOTS[route]}')
    for rank, report in rank_reports.items():
        observed = {key for key, value in report['communications'].items() if value['calls']}
        missing = EXPECTED[route] - observed
        print(f'{route}/rank{rank}: observed={sorted(observed)}, expected={sorted(EXPECTED[route])}')
        assert not missing, f'missing expected collective: {sorted(missing)}'
        if route == 'varlen_cp':
            assert report['kernel_hits'].get('npu_fusion_attention_v3'), 'missing Varlen attention kernel'
            optional = {'TND', 'actual_seq_lengths', 'sparse_mode_7'}
            missing_optional = optional - set(report['kernel_hits'])
            if missing_optional:
                print('  trace_view omitted optional argument text; shape/boundary gate runs below:', sorted(missing_optional))


## 3. 通信账本与理论趋势

两卡 bf16、约 1.7B 参数的数量级账本：

- Varlen+FSDP：参数 shard 约减半，forward/backward 支付 all-gather/reduce-scatter；
- Varlen+CP：Q/K/V 和 output 支付 all-to-all，序列 8192 在两卡分片；

按文档长度估算的 Attention 工作量、逻辑 tensor bytes 和 profiler 的 wire/Transit Size 不是同一个指标。只有当前 CSV/trace 与关闭 profiler 的吞吐都支持时，才能写“加速”。

In [ ]:
def normalized_rates(wall_seconds: float, *, raw_samples: int, input_tokens: int, supervised_tokens: int) -> dict[str, float]:
    if wall_seconds <= 0:
        raise ValueError('wall_seconds must be positive')
    return {
        'raw_sample/s': raw_samples / wall_seconds,
        'input_token/s': input_tokens / wall_seconds,
        'supervised_token/s': supervised_tokens / wall_seconds,
    }

# Fill these only from the matching run's log/DataLoader counters.
# normalized_rates(wall_seconds=..., raw_samples=..., input_tokens=..., supervised_tokens=...)


## 4. 双 rank 关键路径

两组都处理 16,384 个容器位置。`Stage` 取两卡较慢值，但要逐 rank 查看 `Computing` 和 `Communication(Not Overlapped)`，否则会把另一张卡的同步等待误判成链路传输。

## 5. 输入形状、有效 token 与 Varlen 稀疏度

以下 cell 离线重放第五个 yielded batch。它对应训练日志的 Step 5，也对应 Ascend CSV 零起始编号的 `Step=4`；重放本身只读取本地 tokenizer 和 Wordle 数据。Profiler step 用于归因，最终加速比应再用关闭 profiler 的稳定吞吐确认。


In [ ]:
STEP_COLUMNS = ('Stage', 'Computing', 'Communication', 'Communication(Not Overlapped)', 'Free')

def step_times(output: Path) -> dict[str, float]:
    path = output / 'step_trace_time.csv'
    if not path.exists():
        return {}
    record = rows(path)[0]
    return {column: number(record, column) / 1000 for column in STEP_COLUMNS}

STEP_TIMES_BY_RANK = {
    route: {rank: step_times(output) for rank, output in outputs.items()}
    for route, outputs in TRACE_OUTPUTS_BY_RANK.items()
}
for route, rank_timings in STEP_TIMES_BY_RANK.items():
    for rank, timing in rank_timings.items():
        print(f'{route}/rank{rank}', {key: round(value, 3) for key, value in timing.items()})

CRITICAL_RANKS = {
    route: max(rank_timings, key=lambda rank: rank_timings[rank]['Stage'])
    for route, rank_timings in STEP_TIMES_BY_RANK.items()
}
STEP_TIMES = {
    route: rank_timings[CRITICAL_RANKS[route]]
    for route, rank_timings in STEP_TIMES_BY_RANK.items()
}

fsdp = STEP_TIMES['varlen_fsdp']
cp = STEP_TIMES['varlen_cp']
if fsdp and cp:
    print('Raw stage times loaded. Performance comparison waits for the boundary check below.')
    print(f'FSDP Stage={fsdp["Stage"]:.3f} ms, CP Stage={cp["Stage"]:.3f} ms')


In [ ]:
%%bash
set -euo pipefail
mkdir -p outputs/profile_traces
export HF_DATASETS_OFFLINE=1
export HF_DATASETS_CACHE=/tmp/ch8-hf-datasets
python3 -u - 2> outputs/profile_traces/08_workload_replay.stderr.log <<'PY'
import contextlib
import json
import math
from dataclasses import replace
from pathlib import Path

log_path = Path('outputs/profile_traces/08_workload_replay.stdout.log')
with log_path.open('w', encoding='utf-8') as log, contextlib.redirect_stdout(log):
    import torch
    import torchtitan_npu
    from torchtitan.components.loss import IGNORE_INDEX
    from torchtitan_npu.models.qwen3.config_registry import sft_qwen3_1_7b_wordle_tnd

    config = sft_qwen3_1_7b_wordle_tnd()
    config = replace(
        config,
        dataloader=replace(config.dataloader, dataset_path='./assets/data/wordle'),
    )
    tokenizer = config.tokenizer.build(tokenizer_path=config.hf_assets_path)

    def replay_profiled_batch(*, dp_world_size: int, dp_rank: int, seq_len: int):
        loader = config.dataloader.build(
            dp_world_size=dp_world_size,
            dp_rank=dp_rank,
            tokenizer=tokenizer,
            seq_len=seq_len,
            local_batch_size=2,
        )
        iterator = iter(loader)
        for _ in range(5):
            input_dict, labels = next(iterator)

        inputs = input_dict['input']
        positions = input_dict['positions']
        lengths = []
        padding_tokens = 0
        padding_segments = 0
        for sample, sample_positions, sample_labels in zip(inputs, positions, labels):
            starts = sample_positions.eq(0).nonzero(as_tuple=True)[0].tolist()
            if not starts or starts[0] != 0:
                raise ValueError('each packed sequence must start at position 0')
            ends = starts[1:] + [seq_len]
            for start, end in zip(starts, ends):
                length = end - start
                lengths.append(length)
                # positions 决定区间；下面只判断该区间是否为 DataLoader 补出的 padding。
                # EOS 不参与寻找样本边界。
                is_padding = bool(
                    sample[start:end].eq(tokenizer.eos_id).all()
                    and sample_labels[start:end].eq(IGNORE_INDEX).all()
                )
                if is_padding:
                    padding_tokens += length
                    padding_segments += 1

        causal_pairs = sum(length * (length + 1) // 2 for length in lengths)
        dense_pairs = inputs.shape[0] * seq_len * (seq_len + 1) // 2
        ordered_lengths = sorted(lengths)
        return {
            'container_tokens': inputs.numel(),
            'nonpadding_tokens': inputs.numel() - padding_tokens,
            'supervised_tokens': int(labels.ne(IGNORE_INDEX).sum()),
            'segments_with_padding': len(lengths),
            'raw_samples': len(lengths) - padding_segments,
            'padding_tokens': padding_tokens,
            'padding_segments': padding_segments,
            'padding_segment_fraction': padding_segments / len(lengths),
            'singleton_segments': sum(length == 1 for length in lengths),
            'mean_segment_tokens': sum(lengths) / len(lengths),
            'segment_p95_tokens': ordered_lengths[math.ceil(0.95 * len(lengths)) - 1],
            'causal_pairs': causal_pairs,
            'dense_pairs': dense_pairs,
            'varlen_sparsity': 1 - causal_pairs / dense_pairs,
            'max_sample_tokens': max(lengths),
        }

    rank0 = replay_profiled_batch(dp_world_size=2, dp_rank=0, seq_len=4096)
    rank1 = replay_profiled_batch(dp_world_size=2, dp_rank=1, seq_len=4096)
    cp = replay_profiled_batch(dp_world_size=1, dp_rank=0, seq_len=8192)
    sum_keys = (
        'container_tokens', 'nonpadding_tokens', 'supervised_tokens',
        'segments_with_padding', 'raw_samples', 'padding_tokens', 'padding_segments',
        'singleton_segments',
        'causal_pairs', 'dense_pairs',
    )
    fsdp_global = {key: rank0[key] + rank1[key] for key in sum_keys}
    fsdp_global['varlen_sparsity'] = 1 - fsdp_global['causal_pairs'] / fsdp_global['dense_pairs']
    fsdp_global['max_sample_tokens'] = max(rank0['max_sample_tokens'], rank1['max_sample_tokens'])
    fsdp_global['padding_segment_fraction'] = fsdp_global['padding_segments'] / fsdp_global['segments_with_padding']
    fsdp_global['mean_segment_tokens'] = fsdp_global['container_tokens'] / fsdp_global['segments_with_padding']
    report = {
        'profiled_step': 4,
        'fsdp_rank0': rank0,
        'fsdp_rank1': rank1,
        'varlen_fsdp_global': fsdp_global,
        'varlen_cp': cp,
    }

report_path = Path('outputs/profile_traces/08_workload_replay.json')
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
PY


### Trace 交叉验证

把重放结果与 trace shape 交叉验证。这里是硬门槛：数量对不上，说明 trace 来自旧边界实现，后面的耗时只能视为历史记录。


In [ ]:
WORKLOAD_REPLAY = json.loads(
    (OUTPUT_ROOT / 'profile_traces/08_workload_replay.json').read_text(encoding='utf-8')
)

def attention_signature(output: Path) -> dict[str, object]:
    for record in rows(output / 'operator_details.csv'):
        if record['Name'] != 'npu::npu_fusion_attention_v3' or not field(record, 'Input Shapes'):
            continue
        shapes = [item.strip() for item in field(record, 'Input Shapes').split(';') if item.strip()]
        tensor_shapes = [tuple(map(int, item.split(','))) for item in shapes[:3]]
        return {
            'q': tensor_shapes[0], 'k': tensor_shapes[1], 'v': tensor_shapes[2],
            'document_boundaries': int(shapes[-1]),
        }
    return {}

SIGNATURES_BY_RANK = {
    route: {rank: attention_signature(output) for rank, output in outputs.items()}
    for route, outputs in TRACE_OUTPUTS_BY_RANK.items()
}
BOUNDARY_CHECKS = []
for route, rank_signatures in SIGNATURES_BY_RANK.items():
    for rank, signature in rank_signatures.items():
        replay_key = f'fsdp_rank{rank}' if route == 'varlen_fsdp' else 'varlen_cp'
        expected = WORKLOAD_REPLAY[replay_key]['segments_with_padding']
        observed = signature.get('document_boundaries')
        matched = observed == expected
        BOUNDARY_CHECKS.append(matched)
        print(f'{route}/rank{rank}: trace intervals={observed}, current replay intervals={expected}, match={matched}')
TRACE_BOUNDARIES_CURRENT = (
    RANK_COVERAGE_CURRENT and bool(BOUNDARY_CHECKS) and all(BOUNDARY_CHECKS)
)
if not TRACE_BOUNDARIES_CURRENT:
    print('双 rank 覆盖不完整或 trace 与当前样本边界不一致：跳过性能结论，请先重跑 08.03。')
print('Attention signatures:', SIGNATURES_BY_RANK)
SIGNATURES = {route: rank_signatures[min(rank_signatures)] for route, rank_signatures in SIGNATURES_BY_RANK.items()}

fsdp_work = WORKLOAD_REPLAY['varlen_fsdp_global']
cp_work = WORKLOAD_REPLAY['varlen_cp']
print('\nLogical workload for captured Step 4')
print('route | container | non-padding | supervised | raw samples | padding | causal pairs | sparsity')
for name, work in (('FSDP-global', fsdp_work), ('CP', cp_work)):
    print(
        f"{name} | {work['container_tokens']} | {work['nonpadding_tokens']} | "
        f"{work['supervised_tokens']} | {work['raw_samples']} | {work['padding_tokens']} | "
        f"{work['causal_pairs']} | "
        f"{work['varlen_sparsity']:.2%}"
    )

print('\nFSDP fragment geometry')
for rank in (0, 1):
    work = WORKLOAD_REPLAY[f'fsdp_rank{rank}']
    print(
        f"rank{rank}: intervals={work['segments_with_padding']}, padding={work['padding_tokens']} "
        f"({work['padding_segment_fraction']:.1%}), mean segment={work['mean_segment_tokens']:.2f}, "
        f"p95={work['segment_p95_tokens']}, pairs={work['causal_pairs']:,}"
    )

trace_label = 'current' if TRACE_BOUNDARIES_CURRENT else 'historical; do not compare performance'
print(f'\nVarlen Attention device-self totals [{trace_label}]')
for operator in ('aclnnFlashAttentionVarLenScore', 'aclnnFlashAttentionUnpaddingScoreGrad'):
    values = []
    for route, rank_stats in OPERATOR_STATS_BY_RANK.items():
        for rank, stats in rank_stats.items():
            values.append(f"{route}/rank{rank}={stats[operator]['device_self_total_ms']:.3f} ms")
    print(operator + ': ' + ', '.join(values))

if TRACE_BOUNDARIES_CURRENT:
    print('\nCaptured-step normalized throughput')
    for field_name in ('container_tokens', 'nonpadding_tokens', 'supervised_tokens', 'raw_samples'):
        fsdp_rate = fsdp_work[field_name] / (STEP_TIMES['varlen_fsdp']['Stage'] / 1000)
        cp_rate = cp_work[field_name] / (STEP_TIMES['varlen_cp']['Stage'] / 1000)
        print(f'{field_name}: FSDP={fsdp_rate:.1f}/s, CP={cp_rate:.1f}/s, gain={cp_rate / fsdp_rate - 1:.1%}')
else:
    print('\nNormalized throughput skipped: trace must be regenerated with current boundaries.')


## 6. 通信账本

注意：TorchTitan 的 `fsdp` mesh 是 `dp_shard × cp`。因此 CP 路径仍然使用 FSDP 参数分片；AllToAll 是额外通信，不是对 AllGather/ReduceScatter 的替代。若前面的隔离区间检查失败，下面的旧 trace 只能用来确认通信路径，不能用来解释当前性能。


In [ ]:
if not TRACE_BOUNDARIES_CURRENT:
    print('Historical communication trace: topology evidence only; performance attribution is disabled.')

def sum_collectives(engine: dict[str, dict[str, float]], kinds: tuple[str, ...]):
    keys = ('calls', 'zero_payload_calls', 'elapsed_ms', 'wait_ms', 'sync_ms', 'link_transit_ms', 'transit_mb')
    return {key: sum(engine.get(kind, {}).get(key, 0) for kind in kinds) for key in keys}

COMM_BY_RANK = {}
for route, rank_reports in REPORTS_BY_RANK.items():
    COMM_BY_RANK[route] = {}
    for rank, report in rank_reports.items():
        engine = report['communication_engine']
        param = sum_collectives(engine, ('all_gather', 'reduce_scatter'))
        alltoall = sum_collectives(engine, ('all_to_all',))
        COMM_BY_RANK[route][rank] = {'parameter': param, 'all_to_all': alltoall}
        print(f'{route}/rank{rank}: parameter={param}')
        if alltoall['calls']:
            kernel = OPERATOR_STATS_BY_RANK[route][rank]['HcclAlltoAllV']
            print(
                f"  CP AllToAll: calls={alltoall['calls']:.0f}, payload={alltoall['transit_mb']:.3f} MB, "
                f"engine elapsed/wait/link={alltoall['elapsed_ms']:.3f}/"
                f"{alltoall['wait_ms']:.3f}/{alltoall['link_transit_ms']:.3f} ms, "
                f"device kernel sum={kernel['device_self_total_ms']:.3f} ms"
            )

fsdp_param_comm = COMM_BY_RANK['varlen_fsdp'][0]['parameter']
cp_param_comm = COMM_BY_RANK['varlen_cp'][0]['parameter']
cp_alltoall = COMM_BY_RANK['varlen_cp'][0]['all_to_all']
print(
    'CP/FSDP per-rank total payload:',
    f"{(cp_param_comm['transit_mb'] + cp_alltoall['transit_mb']) / fsdp_param_comm['transit_mb']:.2f}x",
)

for route, rank_timings in STEP_TIMES_BY_RANK.items():
    for rank, timing in rank_timings.items():
        overlap = timing['Communication'] - timing['Communication(Not Overlapped)']
        print(
            f"{route}/rank{rank}: communication={timing['Communication']:.3f} ms, "
            f"overlapped={overlap:.3f} ms ({overlap / timing['Communication']:.1%}), "
            f"unoverlapped={timing['Communication(Not Overlapped)']:.3f} ms"
        )

layers = int(cp_alltoall['calls'] // 8)
q_shape = SIGNATURES['varlen_cp']['q']
k_shape = SIGNATURES['varlen_cp']['k']
q_local_mb = math.prod(q_shape) * 2 / 1e6
k_local_mb = math.prod(k_shape) * 2 / 1e6
expected_alltoall_mb = layers * 2 * (q_local_mb / 2 + k_local_mb / 2 + k_local_mb / 2 + q_local_mb / 2)
print(f'AllToAll accounting: {layers} layers, expected={expected_alltoall_mb:.3f} MB, profiled={cp_alltoall["transit_mb"]:.3f} MB')


## 7. 从输出还原事件

先看上一个单元打印的 `match`。只有所有 rank 都是 `True`，以下三类证据才能放在一起解释：

1. 当前 DataLoader 重放得到的样本数、有效 token 和监督 token；
2. trace 中 Varlen kernel 接收的区间数与 Q/K/V shape；
3. 双 rank 的 Stage、计算、通信等待、AllToAll 次数和显存。

仓库中已有的旧 trace 是按 EOS 切分样本时生成的：一条多轮对话被拆成多段，连续 padding 也被拆成大量长度为 1 的段。那些 trace 可以证明当时执行过哪些 kernel 和 collective，却不能评价当前样本边界实现的 Varlen 成本，也不能据此断言 CP 比 FSDP 快。

### 重采结果：不同 Varlen 区间形状确实造成 FSDP 跨 rank 不均衡

Varlen Attention 的执行时间不只取决于每个 rank 有多少 token，还取决于这些 token 被文档边界切成什么形状。FSDP 的两个 data rank 读取不同的 greedy-packed containers；即使两边都有 8192 个 token，文档长度、文档数量、padding 和短片段数量也可能不同。这些差异既会改变实际需要处理的 Attention 区域，也会改变 Varlen kernel 的调度和利用率。因此 FSDP 两卡可能出现不同的 Attention 时间，先完成的 rank 随后在参数 collective 中等待较慢 rank。

CP 的两个 rank 使用同一批 8192 数据和同一份文档边界，只分别处理一半 Attention heads。CP 并不会消除文档碎度；如果这批数据很碎，两张卡都可能变慢。它能消除的是“两个 rank 拿到不同稀疏模式”造成的额外时间差，让两张卡更可能同时到达同步点。

这也解释了为什么不能仅凭“一半 heads”推断 CP 更快。对由独立短文档组成的 packing，CP 每卡处理全部文档的一半 heads，理论计算量与负载均衡时 FSDP 每卡处理一半文档的全部 heads 大致相同；CP 还要额外支付 AllToAll。只有 FSDP 的跨 rank 时间差及其同步等待足够大，能够覆盖 CP 新增通信时，CP 才可能得到端到端收益。

本轮归因按下面顺序完成：先确认两条路线实际装入多少原始样本和有效 token；再比较两个 FSDP rank 的文档布局、Varlen Attention 时间和总 Computing；随后检查先完成 rank 的 collective elapsed/wait 与实际链路传输；最后比较 CP 两卡的时间差，并确认新增 AllToAll 是否被缩短的关键路径抵消。任何结果都只能限定到本次输入分布、两卡拓扑、软件版本和 batch 配置。

### 当前结论

本轮使用 `_positions_v2` 双 rank trace。四个 rank 的区间数全部通过交叉验证：FSDP rank 0/1 为 `11/5`，CP rank 0/1 均为 `16`。因此下面的数字代表当前位置编号实现，不是旧 EOS 分段。

FSDP 两个 rank 的 Varlen forward+backward device-self 合计为 `81.309/117.686 ms`，相差 44.7%；较早完成 Attention 的 rank 0 在 ReduceScatter 中出现更长等待。CP 两个 rank 为 `91.270/91.483 ms`，只差 0.2%，说明共享同一份区间几何确实消除了这一项 Attention 负载差。不过 CP 每 rank 新增 224 次、2818.572 MB 的 AllToAll，使每 rank 总通信 payload 变为 FSDP 的 1.41 倍；采样 step 的慢 rank Stage 也从 `1658.489 ms` 增至 `1788.742 ms`（+7.9%）。

关闭 profiler 后，固定统计 steps 2–12（Step 1 含编译），并对同一批次范围逐批重放 DataLoader：

| 指标 | Varlen+FSDP 4096 | Varlen+CP 8192 | CP 相对 FSDP |
|---|---:|---:|---:|
| median step time | 1.636 s | 1.722 s | +5.3%（更慢） |
| P95 step time | 1.645 s | 1.728 s | +5.0%（更慢） |
| container token/s | 10,025.8 | 9,521.1 | -5.0% |
| non-padding token/s | 7,985.4 | 8,908.2 | +11.6% |
| supervised token/s | 5,897.5 | 6,416.2 | +8.8% |
| raw sample/s | 7.065 | 8.770 | +24.1% |
| 日志 max reserved | 26.74 GiB | 29.60 GiB | +10.7% |

所以不能笼统写“CP 更快”。在本次两卡配置中，CP 的物理 step 和 slot 吞吐更慢、显存更高，但 8192 容器减少了 padding，因而单位 wall time 处理了更多非 padding token、监督 token 和原始样本。raw sample/s 的 24.1% 提升还依赖这 11 个 batch 的长度分布，不是 CP 的固有加速常数。该结果只支持本次系统配置的吞吐权衡，不支持长期收敛排名。

## 进阶课程小结

CP+Varlen 配置若使用 `global_batch_size=64`、`local_batch_size=2` 和 `dp_replicate=1`，梯度累积次数为 `64 / (2 × 1) = 32`。每个 microbatch 包含 16,384 个容器 token，一个 optimizer step 会累计 524,288 个容器 token。端到端 profiling 应覆盖完整 optimizer step，并用关闭 profiler 的稳定 step time 验证吞吐。


In [ ]:
%cd $original_dir


## 练习

1. （判断题）IGNORE_INDEX 只屏蔽 loss，不会让 padding 位置自动跳过 embedding、QKV、MLP 和 Norm 计算。

2. （判断题）trace 中的隔离区间数必须与当前 positions workload 重放一致，这是使用该 trace 做性能结论的 correctness gate。

3. （判断题）communication.json 中 HCCS、SIO、SDMA 等层级可能描述同一 collective，不能不去重就把 Transit Size 全部相加。

4. （单选题）当前结果支持哪种表述？
    A. CP+8192 的物理 step 略慢，但有效 token、监督 token 和 raw sample 吞吐更高
    B. CP 在任何 workload 下都更快
    C. FSDP 不产生任何通信
    D. 单个最快 kernel 可以代表完整 optimizer step

In [ ]:
!cat ./answer/08.04_answer.txt
